In [1]:
# ============================================================
# Notebook : 09_label_leakage.ipynb
# Purpose  : Address the label-leakage / circular-prediction concern by
#            comparing three model specifications for each abnormal category.
#   Model A (full)      : all 10 continuous sensors, concurrent time t.
#   Model B (no-define) : all sensors EXCEPT the channel that defines the
#                         label (e.g. SkinTemperature for ST). Tests whether
#                         high performance merely reproduces the labeling rule.
#   Model C (past->fut) : lag-1 sensors within a session predict the NEXT-step
#                         abnormality. Tests genuine forward-predictive value,
#                         independent of concurrent label-defining signal.
#   Metrics : ROC-AUC, PR-AUC (average precision; baseline = prevalence),
#             Recall & Precision at 0.5, Brier score (calibration).
#   CV      : 5-fold GroupKFold by household (no household in train & test).
# Design   : Single code cell. Grayscale figures, 600 dpi, png+pdf. No path print.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, brier_score_loss)

warnings.filterwarnings("ignore")
try:
    from functools import partialmethod
    import tqdm as _t, tqdm.auto as _ta
    _t.tqdm.__init__ = partialmethod(_t.tqdm.__init__, disable=True)
    _ta.tqdm.__init__ = partialmethod(_ta.tqdm.__init__, disable=True)
except Exception:
    pass

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "serif", "axes.facecolor": "white",
                     "figure.facecolor": "white", "savefig.facecolor": "white",
                     "savefig.dpi": 600, "figure.dpi": 120,
                     "axes.edgecolor": "0.15", "grid.color": "0.85"})
GRAYS = ["0.20", "0.50", "0.78"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")

SENSORS = meta["sensor_cont"]                       # 10 continuous channels
CATS = meta["abnormal_categories"]                  # HR, BR, ST, SP, ER
DEFINE = meta["category_defining_sensor"]           # label-defining channel
CAT_KO = {"HR": "심박이상", "BR": "호흡이상", "ST": "체온이상",
          "SP": "산소포화도이상", "ER": "응급직접신호"}

df = df.sort_values(["session_id", "Timestamp"]).reset_index(drop=True)

# Lag-1 features within session for Model C (past -> future).
lag = df.groupby("session_id")[SENSORS].shift(1)
lag.columns = [f"{c}__lag1" for c in SENSORS]
df_lag = pd.concat([df[["HH_ID", "session_id"] + [f"cat_{c}" for c in CATS]], lag], axis=1).dropna()


def oof_metrics(X, y, groups, seed=42):
    """5-fold GroupKFold out-of-fold probabilities + metrics."""
    y = y.astype(int).values
    groups = groups.values
    n_groups = len(np.unique(groups))
    n_splits = min(5, n_groups)
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y), dtype=float)
    for tr, te in gkf.split(X, y, groups):
        clf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                     random_state=seed, n_jobs=-1)
        clf.fit(X.iloc[tr], y[tr])
        oof[te] = clf.predict_proba(X.iloc[te])[:, 1]
    pred = (oof >= 0.5).astype(int)
    prev = float(y.mean())
    return {
        "prevalence": prev,
        "AUC": roc_auc_score(y, oof) if y.sum() > 0 else np.nan,
        "PR_AUC": average_precision_score(y, oof) if y.sum() > 0 else np.nan,
        "Recall": recall_score(y, pred, zero_division=0),
        "Precision": precision_score(y, pred, zero_division=0),
        "Brier": brier_score_loss(y, oof),
    }


rows = []
for c in CATS:
    ycol = f"cat_{c}"
    define_sensor = DEFINE[c]
    # Model A: all sensors, concurrent
    rows.append({"category": c, "category_ko": CAT_KO[c], "model": "A_full",
                 "n_features": len(SENSORS),
                 **oof_metrics(df[SENSORS], df[ycol], df["HH_ID"])})
    # Model B: drop the label-defining continuous sensor (if it is in the set)
    if define_sensor in SENSORS:
        keep = [s for s in SENSORS if s != define_sensor]
        rows.append({"category": c, "category_ko": CAT_KO[c], "model": "B_no_define",
                     "n_features": len(keep),
                     **oof_metrics(df[keep], df[ycol], df["HH_ID"])})
    else:
        # ER is defined by raw Button/Shout, already excluded from all continuous models.
        rows.append({"category": c, "category_ko": CAT_KO[c], "model": "B_no_define",
                     "n_features": len(SENSORS), "prevalence": float(df[ycol].mean()),
                     "AUC": np.nan, "PR_AUC": np.nan, "Recall": np.nan,
                     "Precision": np.nan, "Brier": np.nan})
    # Model C: lag-1 (past) sensors -> next-step abnormality
    lag_cols = [f"{s}__lag1" for s in SENSORS]
    rows.append({"category": c, "category_ko": CAT_KO[c], "model": "C_past_future",
                 "n_features": len(lag_cols),
                 **oof_metrics(df_lag[lag_cols], df_lag[ycol], df_lag["HH_ID"])})

# Persistence baseline: use the previous-step label as the score for the current
# label. This quantifies how much of Model C reflects short-term autocorrelation
# (state persistence) rather than novel early detection (review 4.1).
from sklearn.metrics import roc_auc_score as _auc, average_precision_score as _ap
for c in CATS:
    if c == "ER":
        continue
    y = df[f"cat_{c}"].astype(int)
    prev = df.groupby("session_id")[f"cat_{c}"].shift(1)
    d = pd.concat([y.rename("y"), prev.rename("p")], axis=1).dropna()
    rows.append({"category": c, "category_ko": CAT_KO[c], "model": "P_persistence",
                 "n_features": 1, "prevalence": float(d.y.mean()),
                 "AUC": _auc(d.y, d.p), "PR_AUC": _ap(d.y, d.p),
                 "Recall": np.nan, "Precision": np.nan, "Brier": np.nan})

res = pd.DataFrame(rows)
for col in ["prevalence", "AUC", "PR_AUC", "Recall", "Precision", "Brier"]:
    res[col] = res[col].astype(float).round(4)
res.to_csv(TAB / "table_label_leakage.csv", index=False, encoding="utf-8-sig")

# -------- Figure: PR-AUC across the three models, per category, grayscale -----
model_order = ["A_full", "B_no_define", "C_past_future"]
model_lbl = {"A_full": "A: all sensors", "B_no_define": "B: drop label sensor",
             "C_past_future": "C: past->future"}
fig, ax = plt.subplots(figsize=(8.0, 4.2))
x = np.arange(len(CATS))
w = 0.26
for j, m in enumerate(model_order):
    vals = [res[(res.category == c) & (res.model == m)]["PR_AUC"].values[0] for c in CATS]
    ax.bar(x + (j - 1) * w, vals, w, label=model_lbl[m], color=GRAYS[j],
           edgecolor="0.15", linewidth=0.7)
# prevalence baseline markers (random PR-AUC = prevalence)
prev = [res[(res.category == c) & (res.model == "A_full")]["prevalence"].values[0] for c in CATS]
ax.plot(x, prev, "k_", markersize=22, markeredgewidth=1.6, label="prevalence (PR baseline)")
ax.set_xticks(x)
ax.set_xticklabels([f"{c}" for c in CATS])
ax.set_ylabel("PR-AUC (average precision)")
ax.set_xlabel("abnormal category")
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
fig.tight_layout()
save_fig(fig, "fig_label_leakage_prauc")
plt.close(fig)

print("=== Label-leakage 3-model comparison (OOF, GroupKFold by household) ===")
show = res[["category", "model", "n_features", "prevalence", "AUC", "PR_AUC", "Recall", "Precision", "Brier"]]
print(show.to_string(index=False))
print("\nSaved: table_label_leakage.csv, fig_label_leakage_prauc (png+pdf)")


=== Label-leakage 3-model comparison (OOF, GroupKFold by household) ===
category         model  n_features  prevalence    AUC  PR_AUC  Recall  Precision  Brier
      HR        A_full          10      0.1449 0.9633  0.7692  0.7665     0.7603 0.0519
      HR   B_no_define           9      0.1449 0.8497  0.4172  0.5246     0.4480 0.1096
      HR C_past_future          10      0.1454 0.9014  0.5876  0.5831     0.6208 0.0837
      BR        A_full          10      0.0701 0.9750  0.7245  0.7231     0.7118 0.0293
      BR   B_no_define           9      0.0701 0.8231  0.2254  0.1448     0.2838 0.0665
      BR C_past_future          10      0.0703 0.8385  0.2663  0.1879     0.3546 0.0634
      ST        A_full          10      0.0470 0.9893  0.7399  0.7543     0.7272 0.0178
      ST   B_no_define           9      0.0470 0.6760  0.0782  0.0148     0.0827 0.0527
      ST C_past_future          10      0.0472 0.9105  0.3827  0.3555     0.4722 0.0362
      SP        A_full          10      0.0057 0